# ECA-LDNet Springer revision — Kaggle T4 workflow

Run the cells in order; no Kaggle Console commands are required. This notebook uses your attached ITS and RESIDE-6K paths and checks paired-file counts before GPU training.

Kaggle documents a maximum 12-hour execution session for CPU/GPU notebooks. Checkpoint files are written at every epoch under `/kaggle/working/revision_runs`. Before a session limit/reset, stop after an epoch, run the export cell, and download the archive from the notebook Output pane. Same-stage recovery from `stageN_latest.keras` with nonzero `--initial-epoch` loads the compiled checkpoint and preserves AdamW slots, iteration count, and its LR schedule. Stage transitions intentionally create a fresh optimizer for the next objective.

The run order is: GPU check → unpack/locate code → exact data-path and pair preflight → one-epoch pilot → validation-only blend/epsilon screen → capacity table → staged training. Keep SOTS out of model selection.


In [ ]:
from pathlib import Path
import os, sys, time, json, shutil, subprocess
import tensorflow as tf
import keras

print('Python:', sys.version)
print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    raise RuntimeError('Enable a GPU accelerator in Notebook Settings before training.')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
INPUT_ROOT = Path('/kaggle/input')
print('Attached inputs:', [p.name for p in INPUT_ROOT.iterdir()])


In [ ]:
# Find the revision code input. Kaggle may expose the uploaded ZIP or its expanded contents.
archives = list(INPUT_ROOT.rglob('eca_ld_net_revision_ablations_.zip'))
if not archives:
    archives = list(INPUT_ROOT.rglob('eca_ldnet_revision.zip'))
if archives:
    BUNDLE = archives[0]
    PROJECT = Path('/kaggle/working/eca_ldnet_revision')
    PROJECT.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(BUNDLE), str(PROJECT))
else:
    roots = []
    for package_dir in INPUT_ROOT.rglob('eca_ldnet'):
        candidate = package_dir.parent
        if (candidate / 'eca_ldnet' / 'train.py').is_file() and (candidate / 'scripts' / 'match_model_capacity.py').is_file():
            roots.append(candidate)
    if not roots:
        raise FileNotFoundError(
            'Attach the Kaggle dataset eca_ldnet_revision under Add Input. '
            'Expected eca_ldnet_revision.zip or expanded eca_ldnet/ and scripts/ folders.'
        )
    PROJECT = roots[0]
    print('Using expanded code input:', PROJECT)
sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)
print('Package:', (PROJECT / 'eca_ldnet').exists(), '| checkpoint:', (PROJECT / 'Models/stage5_final_ema_model.keras').exists())
from eca_ldnet.data import find_dir, collect_pairs


## Attach and identify datasets

Kaggle dataset folder names can differ by account. The next cell searches the attached input names; edit the two paths if it selects the wrong dataset. RESIDE-6K and ITS are training sources. Keep SOTS reserved for final evaluation.


In [ ]:
# Exact attached Kaggle dataset paths supplied for this run.
RESIDE6K_ROOT = Path('/kaggle/input/datasets/kmljts/reside-6k')
ITS_ROOT = Path('/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard')

for label, root in [('RESIDE-6K', RESIDE6K_ROOT), ('ITS', ITS_ROOT)]:
    print(label, 'exists:', root.is_dir(), '| path:', root)
    if root.is_dir():
        print('Top-level contents:', [p.name for p in list(root.iterdir())[:12]])
    else:
        raise FileNotFoundError(f'{label} path is not mounted at {root}. Check Kaggle Add Input and copy the actual directory path.')

RESIDE_HAZY = find_dir(str(RESIDE6K_ROOT), ['train/hazy', 'Train/hazy'])
RESIDE_GT = find_dir(str(RESIDE6K_ROOT), ['train/GT', 'train/gt', 'train/clear'])
ITS_HAZY = find_dir(str(ITS_ROOT), ['hazy', 'Hazy', 'train/hazy'])
ITS_GT = find_dir(str(ITS_ROOT), ['clear', 'GT', 'gt', 'train/clear', 'train/GT'])
print('RESIDE folders:', RESIDE_HAZY, RESIDE_GT)
print('ITS folders:', ITS_HAZY, ITS_GT)
RESIDE_PAIRS = collect_pairs(RESIDE_HAZY, RESIDE_GT, 'RESIDE-6K preflight')
ITS_PAIRS = collect_pairs(ITS_HAZY, ITS_GT, 'ITS preflight')
if not RESIDE_PAIRS or not ITS_PAIRS:
    raise RuntimeError('Pair preflight found zero pairs for a dataset. Do not launch training; inspect folder names and filename matching above.')
print('Preflight passed. Total paired examples:', len(RESIDE_PAIRS) + len(ITS_PAIRS))


## Pilot: one epoch + grouped split + checkpoint serialization

Run this before any long training. A successful pilot verifies GPU use, data discovery, the content-grouped split manifest, Keras serialization, and epoch timing. If the VGG16 pretrained weights cannot download, enable Internet in notebook settings or upload/cache those weights first.


In [ ]:
assert RESIDE6K_ROOT is not None and ITS_ROOT is not None, 'Set the two dataset roots in the cell above.'
RUN_ROOT = Path('/kaggle/working/revision_runs')
PILOT_ARGS = [
    sys.executable, '-m', 'eca_ldnet.train',
    '--reside6k', str(RESIDE6K_ROOT), '--its', str(ITS_ROOT),
    '--out', str(RUN_ROOT), '--stages', '1', '--max-epochs', '1',
    '--seed', '17', '--split-seed', '42', '--variant', 'full', '--batch-size', '8',
    '--preprocessing', 'radiometric', '--physics-blend', '0.08', '--physics-eps', '0.1', '--perceptual-protocol', 'imagenet',
]
start = time.time()
subprocess.run(PILOT_ARGS, cwd=PROJECT, check=True)
pilot_minutes = (time.time()-start)/60
print('Pilot elapsed minutes (one full Stage-1 epoch):', round(pilot_minutes, 2))
print('Rough 140-epoch single-seed estimate at this same epoch time (hours):', round(140*pilot_minutes/60, 2))
print('This rough estimate excludes stage-to-stage speed differences, sweeps, validation/export overhead, and other training variants.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
PILOT_DIR = RUN_ROOT / 'full' / 'seed_17'
PILOT_CHECKPOINT = PILOT_DIR / 'models' / 'eca_ldnet_final.keras'
PILOT_MANIFEST = PILOT_DIR / 'split_manifest.csv'
assert PILOT_CHECKPOINT.exists() and PILOT_MANIFEST.exists()
from eca_ldnet.model import load_pretrained
pilot_model = load_pretrained(str(PILOT_CHECKPOINT))
assert pilot_model.count_params() == 1_481_871, pilot_model.count_params()
probe = pilot_model.predict(__import__('numpy').zeros((1, 32, 32, 3), dtype='float32'), verbose=0)
assert probe.shape == (1, 32, 32, 3) and __import__('numpy').isfinite(probe).all()
print('Saved model reload + 32x32 forward pass: OK')


## Blend/epsilon sensitivity on validation only

This screens both the legacy checkpoint correction (raw CNN output as ASM input) and the stated observed-hazy-input equation on the same fixed validation images. It is inference-only; after choosing candidates, train each finalist because changing the blend can change learned weights. Do not choose from SOTS test scores.


In [ ]:
LEGACY_CHECKPOINT = PROJECT / 'Models' / 'stage5_final_ema_model.keras'
BLEND_ARGS = [
    sys.executable, str(PROJECT / 'scripts' / 'sweep_asm_blend.py'),
    '--model', str(LEGACY_CHECKPOINT), '--manifest', str(PILOT_MANIFEST),
    '--output', str(RUN_ROOT / 'blend_sweep_legacy'), '--preprocessing', 'submitted',
    '--weights', '0', '0.02', '0.05', '0.08', '0.10', '0.20',
    '--epsilons', '0.05', '0.1', '0.2',
]
subprocess.run(BLEND_ARGS, cwd=PROJECT, check=True)
print((RUN_ROOT / 'blend_sweep_legacy' / 'blend_sweep_summary.csv').read_text()[:5000])


## Capacity-matched comparison setup

This searches widths for the full model, SE, CBAM, ECA-only, spatial-only, standard-convolution U-Net, group-convolution U-Net, and a learned residual refinement branch. Use the resulting width multipliers in the training commands so parameter counts are close. MACs and measured latency still need to be reported separately.


In [ ]:
COST_ARGS = [sys.executable, str(PROJECT / 'scripts' / 'match_model_capacity.py'),
             '--output', str(RUN_ROOT / 'model_capacity_matches.csv')]
subprocess.run(COST_ARGS, cwd=PROJECT, check=True)
print((RUN_ROOT / 'model_capacity_matches.csv').read_text())


## Full training

Run one stage per training-cell execution so you can export results before Kaggle’s 12-hour session limit. For a fresh seed, run `[1]` with no resume checkpoint. Then run `[2]`, `[3]`, and `[4]` separately, setting `RESUME_CHECKPOINT` to the previous stage’s `eca_ldnet_final.keras` and exporting after every stage. Use the same seed, split seed (`42`), stage-batch mapping (`8,16,8,16`), and preprocessing across all compared runs. For same-stage recovery, resume from `stageN_latest.keras` with the nonzero `--initial-epoch`; the compiled checkpoint preserves AdamW slots, iteration count, and its LR schedule. Stage transitions intentionally start a fresh optimizer for the next objective.

The main corrected model uses the observed hazy input in the stabilized ASM correction. Use the exact same preprocessing, split seed, stage budgets, and optimizer settings for controls.


The package implements the recovered S1-S4 source objectives. The original Stage 5 recipe is not recoverable from the checked-in notebooks. The training CLI blocks Stage 5 unless you explicitly opt into a new Stage-4-based continuation; never report it as faithful S5 reproduction.


In [ ]:
# EDIT THESE PER RUN. One training run writes under variant/seed_<seed>.
VARIANT = 'no_eca'  # First independent component ablation; each variant starts from fresh weights.
SEED = 43           # Use 43 for the new third full-model seed or an ablation repeat.
STAGES = [1]          # Run one stage per session/export; then set [2], [3], [4] and load the preceding eca_ldnet_final.keras.
STAGE_BATCH_SIZES = [8, 16, 8, 16]  # Recovered from the full seed-17/73 checkpoint schedules; pass the same mapping to every comparison.
WIDTH_MULTIPLIER = 1.0 # Replace with capacity-matched value from model_capacity_matches.csv for controls.
RESUME_CHECKPOINT = None
INITIAL_EPOCH = 0      # For same-stage recovery only: set to completed CSV row count and use stageN_latest.keras.
MAX_EPOCHS = None       # For a stage restart from stageN_latest.keras, use a short segment (e.g. 5-10).

train_args = [
    sys.executable, '-m', 'eca_ldnet.train',
    '--reside6k', str(RESIDE6K_ROOT), '--its', str(ITS_ROOT),
    '--out', str(RUN_ROOT), '--stages', *map(str, STAGES),
    '--seed', str(SEED), '--split-seed', '42', '--variant', VARIANT,
    '--stage-batch-sizes', *map(str, STAGE_BATCH_SIZES),
    '--width-multiplier', str(WIDTH_MULTIPLIER), '--preprocessing', 'radiometric',
    '--physics-blend', '0.08', '--physics-eps', '0.1', '--perceptual-protocol', 'imagenet',
]
if RESUME_CHECKPOINT:
    train_args += ['--resume', str(RESUME_CHECKPOINT)]
if INITIAL_EPOCH:
    train_args += ['--initial-epoch', str(INITIAL_EPOCH)]
if MAX_EPOCHS is not None:
    train_args += ['--max-epochs', str(MAX_EPOCHS)]
start = time.time()
subprocess.run(train_args, cwd=PROJECT, check=True)
print('Training elapsed hours:', round((time.time()-start)/3600, 2))


## Independent controls (run as separate notebook executions)

Run the same stage groups and seed/split for `no_eca`, `pixel_only`, `eca_only`, `no_attention`, `no_physics`, `se`, `cbam`, `pa_eca`, `parallel`, `standard_unet`, `group_unet`, `learned_residual`, and `learnable_blend`. For stage-4 factorial checks, independently add one of `--disable-ema`, `--disable-polish-loss`, or `--disable-late-lr` to the command. For S1-S3 loss ablations, run `--stages 1 2 3` with one of `--loss-ablation charbonnier|ssim|edge|frequency|perceptual`; for S4, run `--stages 4` with one of `--loss-ablation mse|charbonnier|ssim|ms_ssim|vgg19_perceptual`. The `learnable_blend` variant trains a blend scalar constrained to [0, 0.2], initialized at 0.08. Do not start every grid blindly; first use the measured pilot to budget the remaining T4 time.


## Real/near-real evaluation and export

Attach complete O-HAZE / I-HAZE / NH-HAZE sets where available. Keep paired reference metrics distinct from no-reference scores. This repo includes only the smaller paired comparison subsets for O-HAZE, I-HAZE, Haze-1K, and SOTS; cite actual image counts from the run, not benchmark totals unless you attach the complete dataset.

After each stage group, save a Kaggle Version and download the result archive from the notebook Output panel. Never use validation/test images for blend/model selection and final reporting at the same time.

Paired sets use full-reference metrics and image-level bootstrap intervals. Unpaired real-haze sets use NIQE/BRISQUE/MUSIQ/MANIQA only as perceptual proxies; they do not establish fidelity. Edit the mappings in the next cell to the exact attached dataset directories. LPIPS/DISTS and modern NR-IQA use optional `pyiqa`/PyTorch metrics; install them in a separate notebook cell if not present.


In [ ]:
# Optional full-reference/real-image evaluation. Edit only the directory mappings.
FINAL_MODEL = RUN_ROOT / 'full' / 'seed_17' / 'models' / 'eca_ldnet_final.keras'
PAIRED_DATASETS = {
    # 'O-HAZE': {'hazy': '/kaggle/input/o-haze/hazy', 'gt': '/kaggle/input/o-haze/gt'},
    # 'I-HAZE': {'hazy': '/kaggle/input/i-haze/hazy', 'gt': '/kaggle/input/i-haze/gt'},
    # 'NH-HAZE': {'hazy': '/kaggle/input/nh-haze/hazy', 'gt': '/kaggle/input/nh-haze/gt'},
}
UNPAIRED_DATASETS = {
    # 'RTTS': '/kaggle/input/rtts/test/hazy',
}
EVAL_ROOT = RUN_ROOT / 'real_evaluation'
try:
    import pyiqa, torch
    HAS_PYIQA = True
except ImportError:
    HAS_PYIQA = False
    print('pyiqa is not installed. Paired PSNR/SSIM/MS-SSIM will run; install pyiqa in a separate notebook cell for LPIPS/DISTS and NR-IQA.')

for dataset_name, dirs in PAIRED_DATASETS.items():
    args = [sys.executable, str(PROJECT / 'scripts' / 'evaluate_paired.py'),
            '--model', str(FINAL_MODEL), '--hazy-dir', dirs['hazy'], '--gt-dir', dirs['gt'],
            '--dataset', dataset_name, '--output', str(EVAL_ROOT / dataset_name),
            '--preprocessing', 'radiometric', '--native-resolution']
    if HAS_PYIQA:
        args += ['--perceptual', 'lpips', 'dists']
    subprocess.run(args, cwd=PROJECT, check=True)

for dataset_name, hazy_dir in UNPAIRED_DATASETS.items():
    if not HAS_PYIQA:
        print(f'Skipping NR-IQA for {dataset_name}: install pyiqa first.')
        continue
    args = [sys.executable, str(PROJECT / 'scripts' / 'evaluate_unpaired.py'),
            '--model', str(FINAL_MODEL), '--hazy-dir', hazy_dir, '--dataset', dataset_name,
            '--output', str(EVAL_ROOT / dataset_name), '--preprocessing', 'radiometric']
    subprocess.run(args, cwd=PROJECT, check=True)

if not PAIRED_DATASETS and not UNPAIRED_DATASETS:
    print('No real datasets attached/configured; edit the mappings above after attaching data.')


In [ ]:
# Package code, histories, split manifests, and checkpoints for export from /kaggle/working.
RESULT_ARCHIVE = Path('/kaggle/working/eca_ldnet_revision_results.zip')
shutil.make_archive(str(RESULT_ARCHIVE.with_suffix('')), 'zip', root_dir=RUN_ROOT)
print('Result archive:', RESULT_ARCHIVE, '| bytes:', RESULT_ARCHIVE.stat().st_size)
